# Day 063 — Exercise 4: Gated API

Wire `check_rate_limit` into a FastAPI app. The `/ask` endpoint checks the user's plan before calling the process function and returns `429 Too Many Requests` when the daily limit is exceeded.

`initial_usage` lets tests start the counter at any value — so you can test the at-limit case without making 10 actual requests.

In [ ]:
FEATURE_MATRIX = {
    "free": {"basic_chat", "view_history"},
    "pro":  {"basic_chat", "view_history", "advanced_chat", "export", "api_access"},
    "enterprise": {"basic_chat", "view_history", "advanced_chat", "export",
                   "api_access", "white_label", "priority_support"},
}

DAILY_LIMITS = {
    "free":       10,
    "pro":        1_000,
    "enterprise": float("inf"),
}

def check_rate_limit(usage_count: int, plan: str) -> tuple[bool, str]:
    limit = DAILY_LIMITS.get(plan, 0)
    if usage_count >= limit:
        return False, f"Daily limit reached for {plan!r} plan"
    return True, ""


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient


## Task

Implement `build_gated_api(plan='free', process_fn=None, initial_usage=0) -> FastAPI`:

```
GET /plan  → {plan, usage_today, limit}
POST /ask  → {answer, plan, requests_remaining}  or  429
```

- Check rate limit BEFORE calling process_fn
- Return `HTTPException(429)` if blocked
- Increment `usage` only on successful asks
- `requests_remaining = limit - usage` (use `-1` when limit is `float('inf')`)

## Your Implementation

In [ ]:
def build_gated_api(plan: str = "free", process_fn=None,
                    initial_usage: int = 0) -> FastAPI:
    """FastAPI with rate-limited /ask endpoint.

    GET /plan                    → {plan, usage_today, limit}
    POST /ask {prompt: str}      → {answer, plan, requests_remaining}
                                 → 429 if rate limit exceeded
                                 → 422 if prompt is empty

    State:
        plan — fixed for this app instance (test parameter)
        usage — starts at initial_usage, increments on each successful /ask
    process_fn: optional callable(prompt: str) -> str for testing.
    """
    # TODO: create app + state, add /plan and /ask routes, enforce rate limit
    raise NotImplementedError


In [ ]:
def build_gated_api(plan: str = "free", process_fn=None,
                    initial_usage: int = 0) -> FastAPI:
    app    = FastAPI()
    _state = {"plan": plan, "usage": initial_usage}

    class _AskReq(BaseModel):
        prompt: str = Field(min_length=1)

    @app.get("/plan")
    def get_plan():
        lim = DAILY_LIMITS.get(_state["plan"], 0)
        return {"plan": _state["plan"],
                "usage_today": _state["usage"],
                "limit": lim if lim != float("inf") else -1}

    @app.post("/ask")
    def ask(req: _AskReq):
        allowed, reason = check_rate_limit(_state["usage"], _state["plan"])
        if not allowed:
            raise HTTPException(status_code=429, detail=reason)
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        _state["usage"] += 1
        lim = DAILY_LIMITS.get(_state["plan"], 0)
        remaining = (lim - _state["usage"]) if lim != float("inf") else -1
        return {"answer": answer, "plan": _state["plan"],
                "requests_remaining": remaining}

    return app


## Automated checks

In [ ]:
score, total = 0, 6
try:
    from starlette.testclient import TestClient

    # /plan endpoint
    app = build_gated_api(plan="free", process_fn=str.upper)
    c   = TestClient(app, raise_server_exceptions=False)
    rp  = c.get("/plan")
    assert rp.status_code == 200
    p   = rp.json()
    assert p["plan"] == "free" and p["usage_today"] == 0
    score += 1; print("\u2705 GET /plan returns plan and usage_today")

    # successful ask
    r = c.post("/ask", json={"prompt": "hello"})
    assert r.status_code == 200
    assert r.json()["answer"] == "HELLO"
    score += 1; print("\u2705 POST /ask returns processed answer")

    # usage increments
    rp2 = c.get("/plan")
    assert rp2.json()["usage_today"] == 1
    score += 1; print("\u2705 usage_today increments after /ask")

    # rate limit: at-limit app (initial_usage=10 for free plan)
    app2 = build_gated_api(plan="free", process_fn=str.upper, initial_usage=10)
    c2   = TestClient(app2, raise_server_exceptions=False)
    r2   = c2.post("/ask", json={"prompt": "hello"})
    assert r2.status_code == 429, f"Expected 429, got {r2.status_code}"
    score += 1; print("\u2705 429 when free rate limit (10) is reached")

    # pro user with same usage is allowed
    app3 = build_gated_api(plan="pro", process_fn=str.upper, initial_usage=10)
    c3   = TestClient(app3, raise_server_exceptions=False)
    r3   = c3.post("/ask", json={"prompt": "hello"})
    assert r3.status_code == 200
    score += 1; print("\u2705 pro user allowed at usage=10 (limit=1000)")

    # empty prompt → 422
    r4 = c.post("/ask", json={"prompt": ""})
    assert r4.status_code == 422
    score += 1; print("\u2705 empty prompt \u2192 422")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_gated_api(plan: str = "free", process_fn=None,
                    initial_usage: int = 0) -> FastAPI:
    app    = FastAPI()
    _state = {"plan": plan, "usage": initial_usage}

    class _AskReq(BaseModel):
        prompt: str = Field(min_length=1)

    @app.get("/plan")
    def get_plan():
        lim = DAILY_LIMITS.get(_state["plan"], 0)
        return {"plan": _state["plan"],
                "usage_today": _state["usage"],
                "limit": lim if lim != float("inf") else -1}

    @app.post("/ask")
    def ask(req: _AskReq):
        allowed, reason = check_rate_limit(_state["usage"], _state["plan"])
        if not allowed:
            raise HTTPException(status_code=429, detail=reason)
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        _state["usage"] += 1
        lim = DAILY_LIMITS.get(_state["plan"], 0)
        remaining = (lim - _state["usage"]) if lim != float("inf") else -1
        return {"answer": answer, "plan": _state["plan"],
                "requests_remaining": remaining}

    return app
```

**Why HTTP 429?** RFC 6585 defines 429 Too Many Requests specifically for rate limiting. Clients (browsers, SDKs) can detect 429 and implement automatic retry-with-backoff. A 403 Forbidden doesn't signal that the request could succeed later.

</details>